Today's topics:

* how a large language model works
* chatbots, tool calls, and model versions
* training data, copyright, and the law

# Germanium on silicon

Today we will ask a chatbot for the lattice constants we need and look at how it writes its answers.

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/b/bd/Molecular-beam_epitaxy_412_RIBER_system_at_LAAS-CNRS.jpg/960px-Molecular-beam_epitaxy_412_RIBER_system_at_LAAS-CNRS.jpg" alt="A molecular-beam epitaxy system, a stainless steel vacuum chamber with many ports, flanges, cooling lines and cables, standing in a cleanroom laboratory" width=480/>

*Photo by QGravelier, [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0), [via Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Molecular-beam_epitaxy_412_RIBER_system_at_LAAS-CNRS.jpg).*

Above: A [molecular-beam epitaxy](https://en.wikipedia.org/wiki/Molecular-beam_epitaxy) system, like the one from our critical thickness example.
Systems like this grow thin crystal layers, one atomic layer at a time.

Germanium layers grown on silicon wafers are used in fast transistors and in light detectors.
Germanium atoms sit a little farther apart than silicon atoms.

Remember the lattice mismatch from our critical thickness example?

$$f = \frac{a_f - a_s}{a_s}$$

To get $f$ for germanium on silicon we need both lattice constants.

Could we just ask a chatbot for them?

## Let's ask

Here is the request:

> I need some property data for a materials class project. For each of these elements, give me the crystal structure at room temperature, the lattice constant a in angstroms, and the thermal conductivity at room temperature in W/(m K): Fe, Cu, Al, Ni, Ti, W, Si, Ge, Mo, Sn, C, Mn. Give a source for each row. If you do not know a value, leave it blank. Answer as CSV with the columns symbol, structure, lattice_constant, thermal_conductivity, source.

Open [Penn State AI Studio](https://ai.psu.edu/ai-services/ai-studio) and sign in with your Penn State account.

* The model menu lists models from OpenAI, Anthropic, and Google.
* Pick one model and write down its full name.
* Paste the request and send it.

Your reply will likely differ from ours.
The rest of today uses the saved reply below.

## The reply we saved

This is the reply from Claude Haiku 5.5 on November 15, 2026:

```
These values are from memory, not checked against a source. Verify them before they go into the project.

symbol,structure,lattice_constant,thermal_conductivity,source
Fe,BCC (alpha),2.866,80.4,CRC Handbook of Chemistry and Physics (from memory; verify)
Cu,FCC,3.615,401,CRC Handbook of Chemistry and Physics (from memory; verify)
Al,FCC,4.050,237,CRC Handbook of Chemistry and Physics (from memory; verify)
Ni,FCC,3.524,90.9,CRC Handbook of Chemistry and Physics (from memory; verify)
Ti,HCP (alpha),2.951,21.9,CRC Handbook of Chemistry and Physics (from memory; verify); a only, c = 4.686 A
W,BCC,3.165,173,CRC Handbook of Chemistry and Physics (from memory; verify)
Si,Diamond cubic,5.431,149,CRC Handbook of Chemistry and Physics (from memory; verify)
Ge,Diamond cubic,5.658,60.2,CRC Handbook of Chemistry and Physics (from memory; verify)
Mo,BCC,3.147,138,CRC Handbook of Chemistry and Physics (from memory; verify)
Sn,Tetragonal (white tin),5.832,66.6,CRC Handbook of Chemistry and Physics (from memory; verify); a only, c = 3.182 A
C,Hexagonal (graphite),2.464,,Lattice from CRC Handbook (from memory; verify); conductivity left blank because graphite is strongly anisotropic (no single value)
Mn,Cubic (alpha-Mn),8.913,7.81,CRC Handbook of Chemistry and Physics (from memory; verify)

Notes: lattice constants are a only, at about 298 K. Thermal conductivity is at about 300 K. Source strings are general citations, not page-level references. Sources are quoted per row for the CRC Handbook; no page numbers were checked.
```

Every row cites the CRC Handbook.
The reply also says it did not check the handbook.
Even "from memory" is generated text.

The citation names where such numbers are usually printed.
It does not tell us where these came from.

Models can invent values or citations, like a paper that does not exist.
These errors are often called **hallucinations**.
We have not checked this reply yet.

Let's copy the numbers into a table.
Let's leave out the source column and use `np.nan` for the blank:

In [1]:
import numpy as np
import pandas as pd
from io import StringIO  # reads a CSV from text

In [2]:
llm = pd.DataFrame({
    'symbol': ['Fe', 'Cu', 'Al', 'Ni', 'Ti', 'W', 'Si', 'Ge', 'Mo', 'Sn', 'C', 'Mn'],
    'structure': ['BCC (alpha)', 'FCC', 'FCC', 'FCC', 'HCP (alpha)', 'BCC',
                  'Diamond cubic', 'Diamond cubic', 'BCC', 'Tetragonal (white tin)',
                  'Hexagonal (graphite)', 'Cubic (alpha-Mn)'],
    'lattice_constant': [2.866, 3.615, 4.050, 3.524, 2.951, 3.165,
                         5.431, 5.658, 3.147, 5.832, 2.464, 8.913],  # angstroms
    'thermal_conductivity': [80.4, 401, 237, 90.9, 21.9, 173,
                             149, 60.2, 138, 66.6, np.nan, 7.81],  # W/(m K)
})
display(llm)

,symbol,structure,lattice_constant,thermal_conductivity
0,Fe,BCC (alpha),2.866,80.40
1,Cu,FCC,3.615,401.00
2,Al,FCC,4.050,237.00
3,Ni,FCC,3.524,90.90
4,Ti,HCP (alpha),2.951,21.90
5,W,BCC,3.165,173.00
6,Si,Diamond cubic,5.431,149.00
7,Ge,Diamond cubic,5.658,60.20
8,Mo,BCC,3.147,138.00
9,Sn,Tetragonal (white tin),5.832,66.60


Appendix A shows how to run this notebook on your own reply.

# How a language model works

Chatbots like ChatGPT, Copilot, and Claude run on a **large language model** (LLM).

Let's follow one line of a growth recipe through the model.

## Text to tokens

<img src="https://raw.githubusercontent.com/wfreinhart/matse219/main/datasets/llm_primer_1_tokens.png" alt="The phrase 'WSe2 grown by MOCVD at 100 Torr' split into 12 tokens, each with an integer id: W 54, Se 1584, 2 17, grown 20457, by 656, M 391, OC 8695, VD 13195, at 540, a space 220, 100 1353, Torr 122126. The formula and acronym pieces are outlined in orange." width=640/>

The text is cut into **tokens**, and each token gets an id number.

Common words can be one token.
This tokenizer cuts WSe$_2$ into W, Se, and 2.

A dot marks a space at the start of a token.

## Tokens to numbers

<img src="https://raw.githubusercontent.com/wfreinhart/matse219/main/datasets/llm_primer_2_embedding.png" alt="The token 'grown' with id 20457 points to one row of an embedding table. That row, 0.21, -0.63, 0.08, 0.47 and more, is copied out as the token's embedding, a vector of thousands of numbers." width=640/>

Each id picks one row of a learned table.
That row is the token's **embedding**, a vector of thousands of numbers.
The numbers shown are made up.

<img src="https://raw.githubusercontent.com/wfreinhart/matse219/main/datasets/llm_primer_3_tensor.png" alt="The 12 tokens of the phrase as rows of a colored grid, one row per token and thousands of numbers per row, with blue for negative values and orange for positive values. The grid goes into the model." width=640/>

The whole line becomes a grid of numbers, with one row per token.
This grid is what the model works on.

## Similar meanings sit close together

<img src="https://raw.githubusercontent.com/wfreinhart/matse219/main/datasets/llm_primer_4_latent_space.png" alt="A hand-drawn two-dimensional sketch of a latent space with three groups: materials (WSe2, MoS2, graphene), pressure units (Torr, Pa, mbar), and growth methods (MOCVD, CVD, sputtering)." width=640/>

A text-embedding model can place related words and phrases near each other.
This is like a descriptor space for materials.

This sketch illustrates that idea with hand-placed points.
A real space has thousands of dimensions.

Word embeddings trained on materials science abstracts placed known thermoelectrics near promising new ones ([Tshitoyan et al., 2019](https://doi.org/10.1038/s41586-019-1335-8)).

## Attention

<img src="https://raw.githubusercontent.com/wfreinhart/matse219/main/datasets/llm_primer_5_attention.png" alt="The words 'WSe2 grew on sapphire and it' with arcs from each earlier word to 'it'. The arc from WSe2 is thickest, weight 0.60; sapphire 0.14; it to itself 0.16; the others 0.02 to 0.04. The weighted mix becomes the attention output for 'it', which passes through many layers." width=640/>

Each token mixes in information from itself and the tokens before it.
Here "it" draws most on "WSe2", four words back.

The figure groups tokens into words to stay readable.
The numbers on the arcs are made up.
The model repeats this step in many layers.

## Picking the next token

<img src="https://raw.githubusercontent.com/wfreinhart/matse219/main/datasets/llm_primer_6_next_token.png" alt="The final vector of the last token gives a probability for each possible next token: period 34%, 'and' 21%, comma 14%, 'on' 9%, 'using' 6%, out of about 200,000 tokens. One is picked, here 'and', and it fills the empty slot after the input." width=640/>

The model gives a probability to every token it knows.
The probabilities shown are made up.

**Sampling** picks one token at random, weighted by those probabilities.
Here "and" was picked, even though "." was more likely.
So the same request can give a different reply.

## Training and use

<img src="https://raw.githubusercontent.com/wfreinhart/matse219/main/datasets/llm_primer_7_training_vs_use.png" alt="Two panels. Training: the model predicts next-token probabilities, a loss compares them with the real next token, and the weights are adjusted, repeated many times. Use: the same input goes into the model with its weights locked, and it returns a next token." width=640/>

The **weights** are the numbers inside the model, like the parameters of a fit.

**Training:** the model predicts the next token in a huge amount of text.
The **loss** is large when the model gave the real next token a low probability.
The weights are nudged to lower it, many times over.
Further training on instructions and good answers turns the model into an assistant.

**Use:** the weights stay fixed during a chat.
A company may train its next model on chats.

## One token at a time

<img src="https://raw.githubusercontent.com/wfreinhart/matse219/main/datasets/llm_primer_8_autoregression.png" alt="Three steps. In each, the input goes into the model, which returns one token: 'and', then 'then', then 'cooled'. Each new token is appended to the input before the next step." width=640/>

Each new token is added to the input.
The model runs again.
Every number in our table was written this way.

# Writing from weights or looking it up

A chatbot is more than the model:

* instructions from the company, added before our message
* the conversation so far
* sometimes **tools**: web search, a document database, or running code

**Inference from the weights:** the model writes values from what it learned in training.
Nothing is looked up.

**Tool call:** the model asks a tool to search or compute.
Then it writes from the result.

Training includes examples where the right next tokens are a tool request, like `search("silicon lattice constant")`.
The chatbot app sees the request, runs the search or the code, and adds the result to the input.
The model then keeps writing, one token at a time.

The model uses its weights in both cases.

## The same request, two ways

**Inference only.** Our saved reply has no search step:

```
Si,Diamond cubic,5.431,149,CRC Handbook of Chemistry and Physics (from memory; verify)
Ge,Diamond cubic,5.658,60.2,CRC Handbook of Chemistry and Physics (from memory; verify)
```

**With a tool call.** A web search by Claude Opus 5.5 on November 16, 2026, shortened:

```
Searched the web: "silicon germanium lattice constant room temperature angstroms"
  1. The group IV silicon-germanium random alloys...  ioffe.ru/SVA/NSM/Semicond/SiGe/basic.html
  2. Properties of Si, Ge, and GaAs at 300K           eesemi.com/sigegaas.htm
  3. Lattice Constant Importance                      universitywafer.com/lattice-constant.html

Si: 5.43095 angstroms, Ge: 5.64613 angstroms, at 300 K
Source: https://eesemi.com/sigegaas.htm
```

Signs that a lookup or a computation really happened:

* a search or code step, shown by the app apart from the reply
* a source that we open ourselves and that contains the quoted value

A working link alone does not show that the chatbot opened it.
A citation in plain generated text is not one of these signs.
Our saved reply cites the CRC Handbook, and nothing was looked up.

The search found a real page, but its germanium value is not the measured 5.658 angstroms.
A lookup shows where a number came from.
Whether that source is right is still our call.

Code a chatbot writes has not necessarily run.
Code that runs without an error can still compute the wrong thing.

The syllabus says how to use and document AI tools in this course.

## Model names and versions

Our saved reply came from **Claude Haiku 5.5**:

* Claude: the model family, from Anthropic
* Haiku: the size tier, Anthropic's small, fast models; Sonnet and Opus are larger
* 5.5: the version

GPT-6 Sol and GPT-6 Luna in AI Studio are two GPT-6 models from OpenAI.
Penn State describes Luna as the faster option.

A new model means new weights, and often:

* newer training data, with a later cutoff date
* a changed training procedure, including instruction tuning
* a different size

ChatGPT came out in November 2022.
New models now arrive every few months.

Different weights give different probabilities.
A different model can give a different number for the same request.

### [Check your understanding]

Let's run the same request on a very different model in AI Studio.

1. Pick a model far from your first one: a small, fast model if you started with a large one, or the reverse.
   Open a new chat, paste the same request, and write down the model's name.
   * Compare the Sn and C rows from your two models.
   * Which ideas from today explain the differences?
2. Add "Search the web for each value and link the page you used." to the end of the request, in a new chat.
   * Did the model search? How can you tell?
   * Did any values change?

A model that cannot search, or a reply that did not change, is also a result.

# Graphite or diamond?

In our saved reply, carbon is graphite.
What did your two models give?

In [3]:
display(llm[llm['symbol'] == 'C'])

,symbol,structure,lattice_constant,thermal_conductivity
10,C,Hexagonal (graphite),2.464,NaN


The chatbot gave graphite with $a = 2.464$ angstroms ([AIST crystal database](https://unit.aist.go.jp/mmri/ja/organization/db-nomura/common/struc-coord/C%28Graphite%29-c.html)).
Diamond has $a = 3.567$ angstroms ([Hom, Kiszenik, and Post, 1975](https://doi.org/10.1107/S0021889875010965)).

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/f/f0/Graphite-and-diamond-with-scale.jpg/960px-Graphite-and-diamond-with-scale.jpg" alt="A dark graphite specimen on the left and a small translucent diamond crystal on the right, with a 5 mm scale bar" width=480/>

*Photo by Robert M. Lavinsky, [CC BY-SA 3.0](https://creativecommons.org/licenses/by-sa/3.0/), [via Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Graphite-and-diamond-with-scale.jpg).*

Both numbers are right for their own crystal.
Graphite and diamond are two different materials made of carbon.

The request said carbon, not which crystal.
Each reply chose one.
That choice can depend on the model, the input, and any source it retrieved.

## Where a number comes from

The model writes a reply from its learned weights and the current input.

* Values that are common in training text are more likely to come out right.
* The input can hold a pasted paper, a search result, or a database reply.
  We can check a value against that source.

The reply alone does not show where a value came from.

We compare with a source that describes the same material under the same conditions.
The Materials Project gives calculated values for a model crystal at 0 K.
A small difference from it does not tell us which value is wrong.

# Back to germanium on silicon

Let's compute the mismatch from the chatbot's numbers and from a measurement.

`.iloc[0]` takes the single value out of a one-row selection:

In [4]:
si = llm[llm['symbol'] == 'Si']
ge = llm[llm['symbol'] == 'Ge']

a_si = si['lattice_constant'].iloc[0]
a_ge = ge['lattice_constant'].iloc[0]
print('chatbot:', (a_ge - a_si) / a_si)

print('measured:', (5.657820 - 5.430941) / 5.430941)  # Hom et al. (1975), at 25 C

chatbot: 0.04179709077517958
measured: 0.04177526509678531


The measured line uses lattice constants from X-ray diffraction, corrected to 25 degrees Celsius ([Hom, Kiszenik, and Post, 1975](https://doi.org/10.1107/S0021889875010965)).

Both give 0.0418, or 4.2%.
The chatbot was right here.
We report the measured number because we can cite it.

That is almost 50 times the germanium on gallium arsenide mismatch (0.09%) from our critical thickness example.
This larger mismatch limits the thickness of germanium layers on silicon that grow without dislocations.

## Writing it up

Here is how we could report what we did:

> **Data note.** Lattice constants for 12 elements were generated by Claude Haiku 5.5 on November 15, 2026, from the prompt above, and were not used as measurements.
> For carbon the reply gave graphite.
> The mismatch of germanium on silicon, 4.2%, uses the measured lattice constants of Hom et al. (1975).

# Language models and the law

Let's look at the data at each step: training, input, and output.

## Training data

Models are trained on web pages, books, code, and papers, collected at enormous scale.
Much of that text is copyrighted.

* In *Bartz v. Anthropic* (2025), the judge found the training use fair use in this case.
  Downloading and keeping pirated books in a central library was not fair use.
  Anthropic settled for $1.5 billion.
* In *Thomson Reuters v. Ross*, a company trained a non-generative legal search tool on Westlaw's editorial summaries of cases.
  The tool competed with Westlaw.
  The courts found this was not fair use, in the first appeals ruling on AI training (2026).
  The question for language models is still open.

Measured property values are facts, and US copyright does not protect facts (*Feist v. Rural*, 1991).
The text around them, or an original arrangement of them, may be copyrighted.
Licenses and terms of use can also restrict reuse.

## Algorithmic disgorgement

What happens to a model built on data that was collected or used improperly?

In 2021 the US Federal Trade Commission (FTC) alleged that Everalbum ran face recognition on users' photos without their consent.
In its settlement, Everalbum agreed to delete that data **and the models built from users' photos**.
This remedy is called **algorithmic disgorgement**.

Copyright cases ask for it too: *The New York Times* asked the court to destroy models trained on its articles.

We record where every training set came from and under what license.

## What goes in and what comes out

In 2023, Samsung engineers reportedly pasted semiconductor source code into ChatGPT.
Samsung then restricted generative AI tools on company devices.

A company may train its next model on what we type.
A model can memorize and repeat parts of its training text.
So the nondisclosure agreements and export controls from last time apply here too.

Reporting a generated number as a measurement is fabrication.
We say which AI tool we used and what we did with its output.

An AI cannot be named as an inventor (*Thaler v. Vidal*, 2022).
A person who uses AI as a tool can be.
Being an inventor depends on helping to conceive the invention.

## Questions for discussion

1. A company trains a model to predict alloy hardness.
   Its training data include a supplier's test results, shared under a license that forbids using them to train models.
   A court orders the company to stop using the data and the models trained on them.
   Why would deleting the supplier's rows from the training table fail to comply?
   What should the team have recorded about each data source?
2. A student pastes unpublished data from a sponsored project into a chatbot to clean it up.
   Who could see those data, and what agreements could this break?
3. A chatbot suggests a new alloy composition.
   A research group makes it, tests it, and finds it works.
   Can the chatbot be named as an inventor?
   What would we need to know about what each researcher contributed?
4. Measured property values are facts.
   Can we copy a commercial materials database to train a model?
   What besides copyright decides this?

# Summary

* Tokens, embeddings, and attention
* Next-token sampling and autoregression
* Weights, training, and use
* Inference from the weights and tool calls
* Model names and versions
* Training data licenses and algorithmic disgorgement

## Further reading

* Google Machine Learning Crash Course, [Introduction to large language models](https://developers.google.com/machine-learning/crash-course/llm)
* V. Tshitoyan et al., [Unsupervised word embeddings capture latent knowledge from materials science literature](https://doi.org/10.1038/s41586-019-1335-8), *Nature* 571, 95 (2019)
* FTC, [Everalbum order](https://www.ftc.gov/legal-library/browse/cases-proceedings/192-3172-everalbum-inc-matter) (2021), an early case of algorithmic disgorgement

# Appendix A: Using your own reply

To run this notebook on your own reply, set `LIVE_MODE = True`.
Paste the element rows of your reply under the header line, inside the triple quotes.
Leave out your reply's own header row and any text around the table.

Run this cell, then rerun the cells from "Graphite or diamond?" down.
The text describes the saved reply, so your numbers will differ.

In [5]:
LIVE_MODE = False  # True: use your own reply from AI Studio

pasted_reply = """
symbol,structure,lattice_constant,thermal_conductivity,source
"""

if LIVE_MODE:
    # index_col=False drops extra commas at the end of a row
    llm = pd.read_csv(StringIO(pasted_reply), index_col=False)
    # text like '5.43 A' becomes NaN
    llm['lattice_constant'] = pd.to_numeric(llm['lattice_constant'], errors='coerce')
    llm['thermal_conductivity'] = pd.to_numeric(llm['thermal_conductivity'], errors='coerce')
    display(llm)